# 03 — Análisis de la evolución del entrenamiento

#### Objetivo

Analizar cómo evoluciona mi entrenamiento de running a lo largo del tiempo.

#### Fuente

Utilizamos data/processed/running_preparado.csv, generado por
src/actualizar_running.py.

#### Preguntas de análisis

1. ¿Cómo evolucionan los kilómetros y la frecuencia de entrenamiento?
2. ¿Cómo cambian la duración y el ritmo de las carreras?
3. ¿Qué diferencias existen entre entrenamientos cortos y largos?
4. ¿Cómo se distribuyen las distancias y duraciones?
5. ¿Qué contexto aportan las pulsaciones, el desnivel y otras métricas?

#### Criterios

Analizaremos por separado:
- Carrera exterior: modalidad running.
- Carrera en cinta: modalidad treadmill_running.

Empezaremos por las carreras de exterior y después estudiaremos
las de cinta con las mismas preguntas cuando sean aplicables.

No mezclaremos ambas modalidades al interpretar la evolución del ritmo.
Podremos presentar un resumen conjunto de volumen y frecuencia,
identificando la aportación de cada modalidad.

Las métricas avanzadas se analizarán teniendo en cuenta su disponibilidad
y el cambio de reloj.

### Datos personales y reproducción del proyecto

Los ejemplos, resultados y gráficos que se muestren corresponden al historial personal de la autora. No representan datos de otra persona ni resultados universales.

Para reproducir el proyecto con tus entrenamientos, utiliza tu propia cuenta de Garmin y genera tus propios archivos con `src/actualizar_running.py`. El acceso se introduce durante la ejecución; no escribas correo, contraseña, códigos de verificación ni tokens en el código, en las salidas o en archivos publicados.

Los datos originales, los preparados y los archivos de sesión se mantienen locales. Las referencias a actividades y cifras en los notebooks de exploración y calidad documentan la extracción estudiada; no se actualizan automáticamente con un historial diferente. Si se publican resultados o gráficos, su contenido visible puede copiarse.


## 1. Herramientas y carga del historial

Leemos el CSV preparado por el proceso de actualización. Separamos exterior y cinta para trabajar con sus condiciones y métricas por separado. Al actualizar el CSV, es necesario ejecutar el notebook desde el principio.

In [ ]:
# Cargamos las herramientas para tablas y rutas.
import pandas as pd
from pathlib import Path
# Cargamos las herramientas de Matplotlib para crear gráficos.
import matplotlib.pyplot as plt

# Mostramos desde dónde trabaja el notebook.
print("Carpeta de trabajo:", Path.cwd())

In [ ]:
# Localizamos el historial preparado dentro del proyecto.
# Localizamos el CSV desde la raíz del proyecto o desde notebooks.
raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "data/processed/running_preparado.csv").is_file():
    raiz_proyecto = raiz_proyecto.parent
ruta_datos = raiz_proyecto / "data/processed/running_preparado.csv"

# Leemos la tabla e interpretamos fecha_local como fecha.
df = pd.read_csv(ruta_datos,parse_dates=["fecha_local"])

# Creamos tablas diferentes para exterior y cinta.
df_exterior = df[df["modalidad"]=="running"].copy()
df_cinta = df[df["modalidad"]=="treadmill_running"].copy()

# Comprobamos cuántas actividades contiene cada tabla.
n_carreras = len(df)
n_carreras_exterior = len(df_exterior)
n_carreras_cinta = len(df_cinta)
print("Total:", n_carreras)
print("Exterior:", n_carreras_exterior)
print("Cinta:", n_carreras_cinta)


## 2. Visión general de running exterior

Consultamos el número de actividades, el periodo registrado y la distancia acumulada. Los extremos temporales no garantizan un registro continuo ni meses completos.

In [ ]:
# Buscamos la primera y la última fecha de exterior.
fecha_inicio = df_exterior["fecha_local"].min()
fecha_fin = df_exterior["fecha_local"].max()

# Sumamos los kilómetros de todas las carreras de exterior.
kilometros_totales = df_exterior["distance_km"].sum()


# Mostramos el resumen.
print("Carreras:", n_carreras_exterior)
print("Primera fecha:", fecha_inicio)
print("Última fecha:", fecha_fin)
print(f"Kilómetros totales: {kilometros_totales:.2f} km")


## 3. Volumen y frecuencia mensual

Agrupamos por año y mes para estudiar kilómetros, número de sesiones y distancia media por sesión. Completamos los meses sin registros con cero kilómetros y cero sesiones de exterior; esto no implica ausencia de otros deportes. La distancia media queda ausente cuando no hay sesiones. Los meses parciales requieren cautela al compararlos.

In [ ]:
# Extraemos el año y el mes de cada fecha.
# Por ejemplo, 2026-10-31 pasa a pertenecer al periodo 2026-10.
df_exterior["mes"] = df_exterior["fecha_local"].dt.to_period("M")

# Agrupamos las carreras por mes
kilometros_por_mes = (
    df_exterior.groupby("mes")["distance_km"].sum()
)

# Mostramos los kilómetros mensuales con dos decimales.
display(kilometros_por_mes.round(2))

In [ ]:
# Como hay meses que no salen construimos los meses entre la primera y la última carrera
meses_completos = pd.period_range(
    start=df_exterior["mes"].min(),
    end=df_exterior["mes"].max(),
    freq="M"
)

# Colocamos el resumen sobre el calendario.
# Los meses que no salian reciben un cero.
kilometros_por_mes = kilometros_por_mes.reindex(
    meses_completos,
    fill_value=0
)

# Mostramos el resumen completo.
display(kilometros_por_mes.round(2))

In [ ]:
# Agrupamos por mes y con dos metricas.
resumen_mensual =  df_exterior.groupby("mes").agg(
    kilometros=("distance_km","sum"),
    n_carreras=("activityId","size")
)

# Incluimos los meses sin carreras de exterior registradas.
resumen_mensual = resumen_mensual.reindex(
    meses_completos,
    fill_value=0
)

display(resumen_mensual.round(2))

In [ ]:
# Calculamos la distancia media por carrera de cada mes.
# Sustituimos el divisor 0 por un valor ausente para evitar dividir entre cero.
resumen_mensual["km_por_carrera"] = (
    resumen_mensual["kilometros"]
    / resumen_mensual["n_carreras"].replace(0, float("nan"))
)

print(resumen_mensual)


In [ ]:

# Creamos una posición por mes: 0, 1, 2…
posiciones = list(range(len(resumen_mensual)))

# Convertimos los meses en etiquetas de texto.
etiquetas = resumen_mensual.index.astype(str)

# Creamos dos gráficos verticales que comparten el eje de meses.
fig, ejes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 8),
    sharex=True
)

# Gráfico superior: kilómetros registrados por mes.
ejes[0].bar(
    posiciones,
    resumen_mensual["kilometros"],
    color="steelblue"
)
ejes[0].set_title("Volumen y frecuencia mensual — running exterior")
ejes[0].set_ylabel("Distancia (km)")

# Gráfico inferior: número de carreras registradas por mes.
ejes[1].bar(
    posiciones,
    resumen_mensual["n_carreras"],
    color="darkorange"
)
ejes[1].set_ylabel("Número de carreras")
ejes[1].set_xlabel("Mes")

# Colocamos las etiquetas en las mismas posiciones que las barras.
ejes[1].set_xticks(posiciones)
ejes[1].set_xticklabels(etiquetas, rotation=90)

# Mostramos únicamente números enteros en el eje de carreras.
ejes[1].yaxis.set_major_locator(
    plt.MaxNLocator(integer=True)
)

# Añadimos líneas horizontales suaves para facilitar la lectura.
for eje in ejes:
    eje.set_axisbelow(True)
    eje.grid(axis="y", alpha=0.25)

# Ajustamos los márgenes y mostramos el gráfico.
plt.tight_layout()
plt.show()

### Tiempo mensual y ritmo descriptivo

Sumamos el tiempo del cronómetro y calculamos tiempo total dividido entre distancia total. Es un ritmo ponderado por distancia, expresado en minutos decimales por kilómetro. Describe la mezcla de sesiones del mes y no se utilizará como indicador principal de progreso: puede cambiar por el recorrido, la distancia o el objetivo de las sesiones. Los meses sin carreras conservan ritmo ausente.

In [ ]:
# Sumamos los minutos del cronometros de las carreras de cada mes.
minutos_por_mes = df_exterior.groupby("mes")["duration_min"].sum()

# Añadimos los totales al resumen, emparejando por el mes del índice.
# Los meses sin carreras reciben cero minutos registrados.
resumen_mensual["minutos"] = minutos_por_mes.reindex(
    resumen_mensual.index,
    fill_value=0
)


In [ ]:
# Dividimos minutos totales de cada mes entre kms totales.
# En meses sin kms, dejamos el ritmo como ausente.
resumen_mensual["ritmo_min_km"] = (
    resumen_mensual["minutos"]
    / resumen_mensual["kilometros"].replace(0,float("nan")) 
)

# Revisamos el ritmo mensual calculado.
display(resumen_mensual[["minutos", "kilometros", "ritmo_min_km"]].round(2))

## 4. Volumen y frecuencia semanal — exterior

Semanas de lunes a domingo; cero representa ausencia de registros de exterior, no de otros deportes. Mostramos las últimas 12 semanas del historial. La modificación del CSV es una referencia provisional de actualización: copiar o editar el archivo puede cambiarla. La semana abierta en esa fecha es provisional y la primera semana del historial puede tener cobertura parcial. El cierre del calendario no garantiza sincronización completa.

In [ ]:
# Asignamos las actividades a semanas de lunes a domingo.
df_exterior["week"] = df_exterior["fecha_local"].dt.to_period("W-SUN")
resumen_semanal = df_exterior.groupby("week").agg(kilometros=("distance_km", "sum"), n_carreras=("activityId", "size"))
semanas_completas = pd.period_range(df_exterior["week"].min(), df_exterior["week"].max(), freq="W-SUN")
resumen_semanal = resumen_semanal.reindex(semanas_completas, fill_value=0)
# Referencia de actualización del archivo, no día de apertura del notebook.
instante_actualizacion = ruta_datos.stat().st_mtime
fecha_actualizacion = pd.to_datetime(instante_actualizacion, unit="s", utc=True).tz_convert("Europe/Madrid")
fecha_corte = fecha_actualizacion.tz_localize(None).normalize()
fin_semana = resumen_semanal.index.end_time.normalize()
resumen_semanal["en_curso"] = fin_semana >= fecha_corte
print("Referencia de actualización:", fecha_actualizacion)
display(resumen_semanal.tail(12).round(2))
# Usamos posiciones numéricas comunes para barras y etiquetas.
ultimas_semanas = resumen_semanal.tail(12)
colores = []
for en_curso in ultimas_semanas["en_curso"]:
    colores.append("darkorange" if en_curso else "steelblue")
posiciones = list(range(len(ultimas_semanas)))
fig, ejes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
ejes[0].bar(posiciones, ultimas_semanas["kilometros"], color=colores)
ejes[0].set_title("Volumen y frecuencia de exterior — últimas 12 semanas")
ejes[0].set_ylabel("Distancia (km)")
ejes[1].bar(posiciones, ultimas_semanas["n_carreras"], color=colores)
ejes[1].set_ylabel("Número de carreras")
ejes[1].set_xlabel("Semana de lunes a domingo · naranja: abierta al actualizar")
ejes[1].yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ejes[1].set_xticks(posiciones)
ejes[1].set_xticklabels(ultimas_semanas.index.astype(str), rotation=90)
for eje in ejes:
    eje.set_axisbelow(True)
    eje.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Distribución de distancias

Revisamos las estadísticas descriptivas y el número de carreras por kilómetro aproximado. El redondeo se utiliza solo para el recuento visual; las distancias originales permanecen intactas. Una media y una mediana diferentes pueden reflejar asimetría en la distribución.

In [ ]:
# Resumimos las distancias de las carreras de exterior.
resumen_distancias = df_exterior["distance_km"].describe()

display(resumen_distancias.round(2))

In [ ]:
# Redondeamos las distancias al kilómetro más cercano.
# No modificamos la distancia original.
km_redondeados = df_exterior["distance_km"].round().astype(int)

# Contamos las carreras de cada distancia y ordenamos de menor a mayor.
carreras_por_km = km_redondeados.value_counts().sort_index()

# Incluimos todos los kilómetros del rango, también 0 si hay carreras que redondean a 0.
carreras_por_km = carreras_por_km.reindex(
    range(min(1, int(km_redondeados.min())), int(km_redondeados.max()) + 1),
    fill_value=0
)

# Ponemos nombre al indice y mostramos la tabla.
carreras_por_km.index.name = "Distancia aprox (km)"
display(carreras_por_km.to_frame(name="n_carreras"))

In [ ]:
# Creamos el área del gráfico.
fig, ax = plt.subplots(figsize=(12, 5))

# Dibujamos una barra por cada distancia aproximada.
ax.bar(
    carreras_por_km.index,
    carreras_por_km.values,
    color="steelblue"
)

# Mostramos todos los kilómetros del índice en el eje horizontal.
ax.set_xticks(carreras_por_km.index)

# Explicamos qué representa el gráfico.
ax.set_title("Número de carreras por distancia aproximada — exterior")
ax.set_xlabel("Distancia redondeada al kilómetro más cercano")
ax.set_ylabel("Número de carreras")

# Añadimos líneas de referencia y mostramos la figura.
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Distribución de duraciones

Analizamos el tiempo del cronómetro en minutos. Los intervalos tienen una amplitud de 15 minutos y su límite superior se adapta al máximo observado. Cada intervalo incluye el extremo inferior y excluye el superior.

In [ ]:
# Obtenemos las estadísticas descriptivas de la duración en minutos.
resumen_duracion = df_exterior["duration_min"].describe()

# Mostramos el resumen con dos decimales.
display(resumen_duracion.round(2))

In [ ]:
# Elegimos intervalos de 15 minutos.
ancho_intervalo = 15

# Calculamos cuantos intervalos necesitamos para cubrir la duración max.
n_intervalos = (
    int(df_exterior["duration_min"].max()// ancho_intervalo) + 1
)

# Creamos los límites, incluyendo uno superior a la duración máxima.
limites_minutos = [
    i * ancho_intervalo
    for i in range(n_intervalos + 1)
]


# Asignamos cada carrera a su intervalo de duración.
intervalos_duracion = pd.cut(
    df_exterior["duration_min"],
    bins=limites_minutos,
    right=False
)

# Contamos las carreras y ordenamos los intervalos.
carreras_por_duracion = (
    intervalos_duracion.value_counts().sort_index()
)

# Mostramos la tabla de frecuencias.
display(carreras_por_duracion.to_frame(name="n_carreras"))

In [ ]:
# Creamos la figura.
fig, ax = plt.subplots(figsize=(11, 5))

# Dibujamos una barra por intervalo de duración.
carreras_por_duracion.plot.bar(
    ax=ax,
    color="steelblue",
    edgecolor="white"
)

# Añadimos título y etiquetas.
ax.set_title("Distribución de la duración — running exterior")
ax.set_xlabel("Intervalo de duración (minutos)")
ax.set_ylabel("Número de carreras")

# Giramos las etiquetas para facilitar su lectura.
ax.tick_params(axis="x", labelrotation=45)

# Añadimos líneas horizontales y ajustamos los márgenes.
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Evolución mensual de la duración — exterior

La mediana describe el centro de las duraciones y Q1–Q3 su dispersión central. La banda no es un intervalo de confianza. Los cuartiles se interpolan: con pocas sesiones requieren cautela y con una sola coinciden. Los meses sin carreras quedan ausentes y los meses de borde pueden ser parciales.

In [ ]:
# Resumimos las duraciones por sesión en cada mes.
duracion_mensual = df_exterior.groupby("mes").agg(
    n_carreras=("activityId", "size"), mediana_min=("duration_min", "median"),
    q25_min=("duration_min", lambda valores: valores.quantile(0.25)),
    q75_min=("duration_min", lambda valores: valores.quantile(0.75))
).reindex(meses_completos)
# Solo rellenamos el recuento, no las duraciones.
duracion_mensual["n_carreras"] = duracion_mensual["n_carreras"].fillna(0).astype(int)
display(duracion_mensual.round(2))
posiciones = list(range(len(duracion_mensual)))
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(posiciones, duracion_mensual["mediana_min"], marker="o", color="steelblue", label="Mediana")
# fill_between sombrea el intervalo entre ambos cuartiles.
ax.fill_between(posiciones, duracion_mensual["q25_min"].to_numpy(), duracion_mensual["q75_min"].to_numpy(), color="steelblue", alpha=0.2, label="Intervalo Q1–Q3")
ax.set_xticks(posiciones)
ax.set_xticklabels(duracion_mensual.index.astype(str), rotation=90)
ax.set_title("Duración de las sesiones por mes — running exterior")
ax.set_xlabel("Mes")
ax.set_ylabel("Tiempo del cronómetro (minutos)")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Comparación por grupos de distancia

Usamos una convención del proyecto: Short, menos de 5 km; Medium, desde 5 hasta menos de 10 km; Long, 10 km o más. Clasificamos con la distancia sin redondear. Los límites no definen el objetivo ni la intensidad de una sesión. Comparamos recuentos y medianas; cada carrera aporta una observación al ritmo mediano.

In [ ]:
# Definimos los límites: el último incluye distaancias futuras.
limites = [0,5,10,float("inf")]

# Damos nombres los intervarlos.
etiquetas = ["Short (<5 km)", "Medium (5–<10 km)", "Long (≥10 km)"]

# Clasificamos las carreras según su distancia original.
df_exterior["distance_group"] = pd.cut(
    df_exterior["distance_km"],
    bins=limites,
    labels=etiquetas,
    right=False
)

# Contamos las carreras y mantenemos el orden de los grupos.
display(
    df_exterior["distance_group"]
    .value_counts()
    .sort_index()
    .to_frame(name="n_carreras")
)

In [ ]:
# Agrupamos las carreras por distancia y resumimos cada grupo.
comparacion_grupos = df_exterior.groupby(
    "distance_group",
    observed=True,
).agg(
    n_carreras=("activityId","size"),
    distancia_mediana_km=("distance_km","median"),
    duracion_mediana_min=("duration_min","median"),
    ritmo_mediano_min_km=("pace_min_km","median"),
)

# Mostramos el resumen con dos decimales.
display(comparacion_grupos.round(2))

### Variabilidad del ritmo dentro de cada grupo

El diagrama de cajas muestra la mediana y el 50 % central entre Q1 y Q3. Los bigotes alcanzan las observaciones situadas dentro de los límites de 1,5 veces el rango intercuartílico desde los cuartiles. Los puntos exteriores no se consideran errores automáticamente. El ritmo se expresa en minutos decimales y un valor menor indica mayor velocidad.

In [ ]:
# Creamos el área del gráfico.
fig, ax = plt.subplots(figsize=(9, 5))

# Dibujamos la distribución del ritmo en cada grupo de distancia.
df_exterior.boxplot(
    column="pace_min_km",
    by="distance_group",
    ax=ax,
    grid=False
)

# Personalizamos los textos y quitamos el título automático de Pandas.
ax.set_title("Distribución del ritmo según distancia — exterior")
fig.suptitle("")
ax.set_xlabel("Grupo de distancia")
ax.set_ylabel("Ritmo (min/km, decimal)")

# Añadimos referencias horizontales y mostramos el gráfico.
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Contexto temporal de los grupos

Contamos los grupos por año y calculamos porcentajes dentro de cada año. Después comparamos el ritmo mediano por año y grupo, siempre junto al número de sesiones. Esta separación ayuda a explorar la composición temporal, pero no controla el recorrido, el esfuerzo ni el objetivo del entrenamiento. Los años inicial y final pueden ser parciales.

In [ ]:
# Extraemos el año de cada carrera.
anios = df_exterior["fecha_local"].dt.year

# Carreras por año,
carreras_por_anio_grupo = pd.crosstab(
    anios,
    df_exterior["distance_group"]
)

display(carreras_por_anio_grupo)

In [ ]:
# Sumamos las carreras de cada año, recorriendo columnas
totales_por_anio = carreras_por_anio_grupo.sum(axis=1)

# Dividimos cada fila entre su total y convertimos a %.
porcentajes_por_anio = carreras_por_anio_grupo.div(
    totales_por_anio,
    axis=0
)*100

display(porcentajes_por_anio.round(1))


In [ ]:
# Guardamos el año de cada actividad para agrupar fácilmente.
df_exterior["anio"] = df_exterior["fecha_local"].dt.year

# Agrupamos por dos criterios a la vez: año y grupo de distancia.
ritmo_por_anio_grupo = df_exterior.groupby(
    ["anio", "distance_group"],
    observed=True
).agg(
    n_carreras=("activityId", "size"),
    ritmo_mediano_min_km=("pace_min_km", "median")
)

# Mostramos los resultados con dos decimales.
display(ritmo_por_anio_grupo.round(2))

## 10. Ritmo y frecuencia cardiaca — exterior

Cada punto es una sesión con ambas métricas presentes. Un ritmo decimal menor significa mayor velocidad. Los años aportan contexto, pero no controlan distancia, recorrido, esfuerzo, condiciones ni cambios de dispositivo. Menor pulso no implica automáticamente mejora.

In [ ]:
# Seleccionamos observaciones con ritmo y pulso disponibles.
df_ritmo_pulso = df_exterior.dropna(subset=["pace_min_km", "averageHR"]).copy()
print("Carreras de exterior:", len(df_exterior))
print("Con ritmo y pulso:", len(df_ritmo_pulso))
print("Sin alguno de esos datos:", len(df_exterior) - len(df_ritmo_pulso))
fig, ax = plt.subplots(figsize=(10, 6))
# Dibujamos una serie de puntos por año.
for anio, carreras_anio in df_ritmo_pulso.groupby(df_ritmo_pulso["fecha_local"].dt.year):
    ax.scatter(carreras_anio["pace_min_km"], carreras_anio["averageHR"], label=str(anio), alpha=0.6, s=30)
ax.set_title("Ritmo y frecuencia cardiaca media por año — exterior")
ax.set_xlabel("Ritmo (min/km, decimal) · más rápido hacia la izquierda")
ax.set_ylabel("Frecuencia cardiaca media (latidos/min)")
ax.legend(title="Año")
ax.set_axisbelow(True)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Comparación exploratoria a ritmos próximos

Seleccionamos entre 5:45 y 6:00 min/km, incluidos los límites: es una elección exploratoria, no fisiológica. Mostramos tamaños de grupo y contexto de distancia y ascenso. Los metros acumulados de ascenso por km no son la pendiente de cada tramo ni describen toda la dificultad. La unión empareja por año, no por posición.

In [ ]:
# between selecciona valores entre ambos límites.
ritmo_similar = df_ritmo_pulso["pace_min_km"].between(5.75, 6.00)
df_comparable = df_ritmo_pulso.loc[ritmo_similar].copy()
comparacion_pulso = df_comparable.groupby("anio").agg(
    n_carreras=("activityId", "size"), ritmo_mediano=("pace_min_km", "median"),
    pulso_mediano=("averageHR", "median"), distancia_mediana_km=("distance_km", "median")
)
df_comparable["ascenso_m_por_km"] = df_comparable["elevationGain"] / df_comparable["distance_km"]
# count cuenta solo mediciones presentes; size cuenta todas las sesiones.
revision_desnivel = df_comparable.groupby("anio").agg(
    con_desnivel=("ascenso_m_por_km", "count"),
    ascenso_mediano_m_por_km=("ascenso_m_por_km", "median")
)
comparacion_contextual = comparacion_pulso.join(revision_desnivel)
display(comparacion_contextual.round(2))

## 11. Ascenso por kilómetro y ritmo — exterior

Usamos sesiones con datos disponibles. Normalizar el ascenso por distancia facilita describir recorridos de distinta longitud, pero no los hace equivalentes. La asociación visual mezcla fechas, esfuerzos y recorridos y no aísla efectos causales.

In [ ]:
# Seleccionamos observaciones completas para estas variables.
df_ritmo_desnivel = df_exterior.dropna(subset=["distance_km", "pace_min_km", "elevationGain"]).copy()
df_ritmo_desnivel["ascenso_m_por_km"] = df_ritmo_desnivel["elevationGain"] / df_ritmo_desnivel["distance_km"]
print("Carreras con datos:", len(df_ritmo_desnivel))
print("Excluidas por vacíos:", len(df_exterior) - len(df_ritmo_desnivel))
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df_ritmo_desnivel["ascenso_m_por_km"], df_ritmo_desnivel["pace_min_km"], alpha=0.5, s=30)
ax.set_title("Ascenso por kilómetro y ritmo — running exterior")
ax.set_xlabel("Ascenso acumulado por kilómetro (m/km)")
ax.set_ylabel("Ritmo (min/km, decimal)")
ax.set_axisbelow(True)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 12. Running en cinta — descripción y volumen mensual

La propietaria ajusta la distancia según la pantalla de la cinta. El ritmo deriva de esa distancia y del cronómetro, no es una medición independiente de la cinta. Mantenemos el análisis separado del exterior. El calendario conjunto incluye meses exclusivos de cualquiera de las modalidades.

In [ ]:
# Describimos el alcance y las distribuciones de cinta.
print("Carreras en cinta:", len(df_cinta))
print("Primera fecha:", df_cinta["fecha_local"].min())
print("Última fecha:", df_cinta["fecha_local"].max())
print(f"Kilómetros registrados: {df_cinta['distance_km'].sum():.2f}")
display(df_cinta[["distance_km", "duration_min", "pace_min_km"]].describe().round(2))
# Creamos el calendario desde ambas modalidades.
meses = df["fecha_local"].dt.to_period("M")
calendario = pd.period_range(meses.min(), meses.max(), freq="M")
df_cinta["mes"] = df_cinta["fecha_local"].dt.to_period("M")
resumen_cinta_mensual = df_cinta.groupby("mes").agg(
    kilometros=("distance_km", "sum"), n_carreras=("activityId", "size"), minutos=("duration_min", "sum")
).reindex(calendario, fill_value=0)
display(resumen_cinta_mensual.round(2))

## 13. Volumen mensual conjunto por modalidad

La barra apilada suma los kilómetros registrados, manteniendo visible la contribución de cada modalidad. No mezcla ritmos ni compara rendimiento. Los meses de borde pueden ser parciales y cero significa ausencia de registros.

In [ ]:
# unstack convierte las modalidades del índice en columnas.
kilometros_modalidad = df.groupby([meses, "modalidad"])["distance_km"].sum()
volumen_modalidades = kilometros_modalidad.unstack(fill_value=0).reindex(calendario, fill_value=0)
volumen_modalidades = volumen_modalidades.reindex(columns=["running", "treadmill_running"], fill_value=0)
display(volumen_modalidades.round(2))
posiciones = list(range(len(volumen_modalidades)))
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(posiciones, volumen_modalidades["running"], label="Exterior", color="steelblue")
# bottom coloca la cinta encima del kilometraje de exterior.
ax.bar(posiciones, volumen_modalidades["treadmill_running"], bottom=volumen_modalidades["running"], label="Cinta", color="darkorange")
ax.set_xticks(posiciones)
ax.set_xticklabels(volumen_modalidades.index.astype(str), rotation=90)
ax.set_title("Kilómetros mensuales registrados por modalidad")
ax.set_xlabel("Mes")
ax.set_ylabel("Distancia registrada (km)")
ax.legend(title="Modalidad")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 14. Alcance y actualización

El análisis es descriptivo: no demuestra mejoras fisiológicas ni efectos causales. Las métricas opcionales se interpretan con su disponibilidad y el cambio de reloj como contexto.

Para actualizar: ejecutar `src/actualizar_running.py`, reiniciar el kernel y ejecutar todas las celdas. Las tablas y gráficos se recalculan. Las explicaciones no contienen conclusiones numéricas fijas. La referencia de modificación del CSV es provisional hasta guardar metadatos explícitos de extracción.